## Обучаем (несколько эпох) ResNet18 на новом датасете

In [1]:
import sys
sys.path.append('../..')

from pathlib import Path
import os
from glob import glob

import cv2
from PIL import Image
import pandas as pd
import numpy as np

from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import pytorch_lightning as pl
from torch.utils.data import DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

from src.model.baseline_cnn import LitCNN
from src.model.resnet_18 import LitResNet18
from src.model.dataset import ImageDataset
from src.model.classifier import LitClassifier

from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

from tqdm import tqdm
tqdm.pandas()

%load_ext autoreload
%autoreload 2

In [2]:
# Изображения хранятся в директории так, что каждой модели кроссовок соответствует
# своя папка. Чтобы можно было рассчитывать статистики, мы собираем в датафрейм относительный
# путь до каждого из изображений, и рассчитываем агрегаты на основе этого датафрейма
from src.data.utils.eda_utils import directory_to_dataframe

path_to_dataset = Path("../../data/03_yolo_preprocessed/search-dataset-images")
df = directory_to_dataframe(path_to_dataset)
df.head()

,path,sneaker_class
0,Vans Кеды Upland/clothing_0_264.jpeg,Vans Кеды Upland
1,Vans Кеды Upland/clothing_0_57.jpeg,Vans Кеды Upland
2,Vans Кеды Upland/orig_45.jpeg,Vans Кеды Upland
3,Vans Кеды Upland/clothing_0_0.jpeg,Vans Кеды Upland
4,Vans Кеды Upland/clothing_0_233.jpeg,Vans Кеды Upland


In [3]:
# train_df_pre, test_df = split_train_test_by_real(df)

# display(train_df_pre.head(), train_df_pre.shape)
# display(test_df.head(), test_df.shape)

# train_df_pre.to_csv('../../data/03_yolo_preprocessed/train_images.csv')
# test_df.to_csv('../../data/03_yolo_preprocessed/test_images.csv')

In [4]:
train_df_pre = pd.read_csv('../../data/03_yolo_preprocessed/train_images.csv')
display(train_df_pre.head(), train_df_pre.shape)

test_df = pd.read_csv('../../data/03_yolo_preprocessed/test_images.csv')
display(test_df.head(), test_df.shape)

,Unnamed: 0,path,sneaker_class,corrupted_flg
0,0,Vans Кеды Upland/clothing_0_264.jpeg,Vans Кеды Upland,0
1,1,Vans Кеды Upland/clothing_0_57.jpeg,Vans Кеды Upland,0
2,2,Vans Кеды Upland/orig_45.jpeg,Vans Кеды Upland,0
3,3,Vans Кеды Upland/clothing_0_0.jpeg,Vans Кеды Upland,0
4,4,Vans Кеды Upland/clothing_0_233.jpeg,Vans Кеды Upland,0


(10965, 4)

,Unnamed: 0,path,sneaker_class,corrupted_flg
0,14,Vans Кеды Upland/clothing_0_168_real.jpeg,Vans Кеды Upland,0
1,32,Vans Кеды Upland/clothing_0_215_real.jpeg,Vans Кеды Upland,0
2,44,Vans Кеды Upland/orig_216_real.jpeg,Vans Кеды Upland,0
3,80,Vans Кеды Upland/shoe_3_100_real.jpeg,Vans Кеды Upland,0
4,87,Vans Кеды Upland/clothing_0_277_real.jpeg,Vans Кеды Upland,0


(300, 4)

In [5]:
# train_df_pre, test_df = split_train_test_by_real(df)

In [6]:
train_df, val_df = train_test_split(
    train_df_pre,
    test_size=0.2,
    stratify=train_df_pre["sneaker_class"],
    random_state=42
)

display(train_df.head(), train_df.shape)
display(val_df.head(), val_df.shape)
display(test_df.head(), test_df.shape)

,Unnamed: 0,path,sneaker_class,corrupted_flg
2195,2288,Nike Кеды Dunk Low Retro/clothing_0_103.jpeg,Nike Кеды Dunk Low Retro,0
10557,10855,PUMA Кроссовки Puma Morphic/orig_111.jpeg,PUMA Кроссовки Puma Morphic,0
7299,7477,Kari Кроссовки/clothing_0_190.jpeg,Kari Кроссовки,0
4103,4230,Reebok Кроссовки CLASSIC LEATHER/clothing_0_43...,Reebok Кроссовки CLASSIC LEATHER,0
2097,2188,Nike Кеды Dunk Low Retro/clothing_0_86.jpeg,Nike Кеды Dunk Low Retro,0


(8772, 4)

,Unnamed: 0,path,sneaker_class,corrupted_flg
7461,7639,Vans Кеды Knu Skool/clothing_0_98.jpeg,Vans Кеды Knu Skool,0
2486,2591,Reebok Кроссовки CLASSIC NYLON/orig_291.jpeg,Reebok Кроссовки CLASSIC NYLON,0
4889,5027,Nike Кроссовки AIR MAX 90/clothing_0_12.jpeg,Nike Кроссовки AIR MAX 90,0
2655,2762,Under Armour Кроссовки UA CHARGED SPEED SWIFT/...,Under Armour Кроссовки UA CHARGED SPEED SWIFT,0
231,240,Vans Кеды Upland/clothing_0_62.jpeg,Vans Кеды Upland,0


(2193, 4)

,Unnamed: 0,path,sneaker_class,corrupted_flg
0,14,Vans Кеды Upland/clothing_0_168_real.jpeg,Vans Кеды Upland,0
1,32,Vans Кеды Upland/clothing_0_215_real.jpeg,Vans Кеды Upland,0
2,44,Vans Кеды Upland/orig_216_real.jpeg,Vans Кеды Upland,0
3,80,Vans Кеды Upland/shoe_3_100_real.jpeg,Vans Кеды Upland,0
4,87,Vans Кеды Upland/clothing_0_277_real.jpeg,Vans Кеды Upland,0


(300, 4)

In [7]:
train_paths = train_df["path"].tolist()
val_paths   = val_df["path"].tolist()
test_paths  = test_df["path"].tolist()

train_labels = train_df["sneaker_class"].tolist()
val_labels   = val_df["sneaker_class"].tolist()
test_labels  = test_df["sneaker_class"].tolist()

In [8]:
# Аугментация и приведение всех изображений к одному масштабу

train_tfms = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2(),
])

val_tfms = A.Compose([
    A.Resize(224, 224),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2(),
])

In [9]:
train_dataset = ImageDataset(
    base_path=path_to_dataset,
    images_path=train_paths,
    labels=train_labels,
    augmenter=train_tfms
)

val_dataset = ImageDataset(
    base_path=path_to_dataset,
    images_path=val_paths,
    labels=val_labels,
    augmenter=val_tfms
)

test_dataset = ImageDataset(
    base_path=path_to_dataset,
    images_path=test_paths,
    labels=test_labels,
    augmenter=val_tfms
)

In [10]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=8,
    pin_memory=False,
    persistent_workers=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=8,
    pin_memory=False,
    persistent_workers=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=8,
    pin_memory=False,
    persistent_workers=True,
)

In [11]:
callbacks = [
    ModelCheckpoint(
        monitor="val_loss",
        save_top_k=1,
        mode="min"
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=3
    )
]

In [ ]:
from huggingface_hub import login

login("")

import os
os.environ["HF_HUB_DISABLE_SSL_VERIFICATION"] = "1"

In [13]:
# обучаем голову

model = LitClassifier(
    model_name="efficientnet_b0",
    num_classes=df["sneaker_class"].nunique(),
    lr=1e-3,
    freeze_backbone=True
)

trainer = pl.Trainer(
    max_epochs=5,
    callbacks=callbacks
)

trainer.fit(model, train_loader, val_loader)

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [ ]:
# уменьшаем lr и дообучаем все

model.unfreeze()
model.lr = 1e-4

trainer = pl.Trainer(
    max_epochs=10,
    callbacks=callbacks
)

trainer.fit(model, train_loader, val_loader)